<a href="https://colab.research.google.com/github/CelesteBox/Lunes_9AM/blob/main/RAG_Lunes9am_ONEAlura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
# ==========================================
# INSTALACIÓN DE LIBRERÍAS
# ==========================================

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pypdf

In [32]:
!pip install -q sentence-transformers


In [76]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
import os

os.environ["GOOGLE_API_KEY"] = "TU_API_KEY_AQUI"

In [36]:
loader = PyPDFDirectoryLoader("sample_data/data/raw")

documents = loader.load()

print(len(documents))

248


In [37]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

813


In [38]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings locales listos.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings locales listos.


In [39]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("Base vectorial creada.")

Base vectorial creada.


In [70]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("Retriever listo.")

Retriever listo.


In [77]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [71]:
pregunta = "¿Cómo define la UNESCO la inteligencia artificial?"

docs = retriever.invoke(pregunta)

print(len(docs))

5


In [72]:
for i, doc in enumerate(docs):
    print("="*80)
    print(i + 1)
    print(doc.metadata)

1
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2022-05-02T16:20:43+02:00', 'moddate': '2025-04-29T07:16:05+02:00', 'nbpageulisurl': '45', 'trapped': '/False', 'source': 'sample_data/data/raw/UNESCO_Ethics_of_AI_Es.pdf.pdf', 'total_pages': 45, 'page': 40, 'page_label': '41'}
2
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2022-05-02T16:20:43+02:00', 'moddate': '2025-04-29T07:16:05+02:00', 'nbpageulisurl': '45', 'trapped': '/False', 'source': 'sample_data/data/raw/UNESCO_Ethics_of_AI_Es.pdf.pdf', 'total_pages': 45, 'page': 41, 'page_label': '42'}
3
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2022-05-02T16:20:43+02:00', 'moddate': '2025-04-29T07:16:05+02:00', 'nbpageulisurl': '45', 'trapped': '/False', 'source': 'sample_data/data/raw/UNESCO_Ethics_of_AI_Es.pdf.pdf', 'total_pages': 45, 'page': 44, 'page_label': '45'}
4
{'

In [73]:
contexto = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [74]:
prompt = f"""
Sos el asistente Lunes 9 a.m.

Respondé solamente utilizando el contexto.

Si la respuesta no aparece, decí que no está disponible.

CONTEXTO

{contexto}

PREGUNTA

{pregunta}
"""

In [75]:
respuesta = llm.invoke(prompt)

print(respuesta.content)

La definición de inteligencia artificial no está disponible en el contexto proporcionado.
